In [2]:
import torch
from torch.utils import data
from torchvision import transforms as T

from src.dataloaders.dataloader_for_CNN import mavDataLoader, mavDatasetCNN_3D, SequenceBatchSampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Обучение на:', device, sep=' ')

transform = T.Compose([
    T.Resize((320, 192))
])
dataset = mavDatasetCNN_3D('datasets/euroc_mav', transform, device='cpu', batchsize=8, hidden_size=0, lst_of_datasets=['mav0_easy1']) # Сразу формируем все массивы на GPU

groups = dataset.batch_groups.copy()

train_size = int(0.8 * len(groups))
train_groups = groups[:train_size]
test_groups = groups[train_size:]


train_sampler = SequenceBatchSampler(train_groups)
test_sampler = SequenceBatchSampler(test_groups)

train_data = data.DataLoader(dataset, batch_sampler=train_sampler, num_workers=6, pin_memory=True)
test_data = data.DataLoader(dataset, batch_sampler=test_sampler, num_workers=6, pin_memory=True)

Обучение на: cuda


In [3]:
from src.geometry.RotationTorch import RotationTorch as RT 
from src.geometry.PoseTorch import PoseTorch as PT 

In [4]:
dt = iter(train_data)
x, y, T_m = next(dt)

In [5]:
y

tensor([[ 3.7278e-03, -2.9076e-06, -1.4995e-03,  1.0389e-03, -1.5391e-03,
          6.1206e-04],
        [ 3.7516e-03,  1.6501e-06, -1.4949e-03,  1.0831e-03, -1.5670e-03,
          6.5200e-04],
        [ 3.7687e-03,  9.4976e-06, -1.4929e-03,  1.1394e-03, -1.5836e-03,
          7.2510e-04],
        [ 3.7794e-03,  1.7016e-05, -1.4881e-03,  1.1990e-03, -1.5807e-03,
          7.6863e-04],
        [ 3.7815e-03,  2.2685e-05, -1.4851e-03,  1.2347e-03, -1.5771e-03,
          7.6737e-04],
        [ 3.7800e-03,  2.6029e-05, -1.4826e-03,  1.2548e-03, -1.5731e-03,
          7.5644e-04],
        [ 3.7744e-03,  2.6634e-05, -1.4794e-03,  1.2440e-03, -1.5585e-03,
          7.5525e-04],
        [ 3.7671e-03,  2.5957e-05, -1.4750e-03,  1.2243e-03, -1.5533e-03,
          7.6157e-04]], dtype=torch.float64)

In [6]:
T_m

tensor([[ 1.9823, -1.7465,  5.4172,  0.3647,  1.9717,  0.1958],
        [ 1.9798, -1.7495,  5.4164,  0.3645,  1.9703,  0.1974],
        [ 1.9773, -1.7527,  5.4154,  0.3643,  1.9689,  0.1991],
        [ 1.9746, -1.7561,  5.4144,  0.3641,  1.9674,  0.2010],
        [ 1.9719, -1.7598,  5.4135,  0.3639,  1.9660,  0.2029],
        [ 1.9691, -1.7634,  5.4125,  0.3636,  1.9645,  0.2048],
        [ 1.9663, -1.7670,  5.4116,  0.3635,  1.9631,  0.2068],
        [ 1.9634, -1.7706,  5.4107,  0.3633,  1.9617,  0.2087]],
       dtype=torch.float64)

In [9]:
pose1, pose2 = T_m[0], T_m[1]

y_p = (PT.from_lie(pose1).inv() * PT.from_lie(pose2)).as_lie()

In [8]:
y[0]

tensor([ 3.7278e-03, -2.9076e-06, -1.4995e-03,  1.0389e-03, -1.5391e-03,
         6.1206e-04], dtype=torch.float64)

In [11]:
PT.from_lie(pose1) * PT.from_lie(y[0])

PoseTorch(R=RotationTorch(_q=tensor([ 0.5346, -0.1530, -0.8270, -0.0829], dtype=torch.float64)), t=tensor([ 4.6882, -1.7868,  0.7873], dtype=torch.float64))

In [12]:
PT.from_lie(pose2)

PoseTorch(R=RotationTorch(_q=tensor([ 0.5346, -0.1530, -0.8270, -0.0829], dtype=torch.float64)), t=tensor([ 4.6882, -1.7868,  0.7873], dtype=torch.float64))